[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca-Ferrera/itsel_course/blob/main/content/01_esempio_esplorazione_dati.ipynb) [![Apri in JupyterLite](https://jupyterlite.rtfd.io/en/latest/_static/badge.svg)](https://luca-ferrera.github.io/itsel_course/lab/index.html?path=01_esempio_esplorazione_dati.ipynb)

Puoi usare questo notebook in **Google Colab** (serve un account Google) oppure in **JupyterLite** (nessun account, funziona nel browser).

In Colab, prima di modificare il notebook, scegli **File → Salva una copia in Drive**.

# Esploriamo dati reali: COVID-19 in Italia

Benvenuta/o! Un **notebook** è una pagina composta da **celle**: alcune contengono testo, altre istruzioni in Python.

Per eseguire una cella, selezionala e premi **Shift+Invio**. Se qualcosa non funziona, dal menu **Run / Esegui** scegli **Esegui tutto**.

In questo laboratorio il lavoro resta nel browser. Al termine salva una copia del tuo lavoro con **File → Scarica / Download**: così potrai conservarla o consegnarla.

## 1. Prepariamo gli strumenti

Esegui la cella seguente una sola volta. Nel laboratorio online prepara le librerie necessarie; in Google Colab sono già disponibili.

In [ ]:
import sys
if "pyodide" in sys.modules:   # siamo in JupyterLite
    %pip install -q pandas matplotlib scikit-learn

## 2. Carichiamo i dati

Useremo dati pubblici sull'andamento nazionale COVID-19. Nel laboratorio viene usata la copia inclusa; in Colab, se il file non c'è, viene scaricato dalla fonte pubblica. `df` è la nostra tabella di dati.

In [ ]:
import os
import pandas as pd
PERCORSO_LOCALE = "data/dpc-covid19-ita-andamento-nazionale.csv"
URL = "https://raw.githubusercontent.com/pcm-dpc/COVID-19/master/dati-andamento-nazionale/dpc-covid19-ita-andamento-nazionale.csv"
df = pd.read_csv(PERCORSO_LOCALE if os.path.exists(PERCORSO_LOCALE) else URL, parse_dates=["data"])
df

## 3. Guardiamo la tabella

Ogni riga descrive una data; ogni colonna descrive una misura. Proviamo alcune domande semplici.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.columns

Scegliamo poche colonne. `describe()` riassume i numeri: conteggio, media, minimo e massimo.

In [ ]:
colonne = ["data", "ricoverati_con_sintomi", "terapia_intensiva", "totale_positivi"]
df[colonne].head()

In [ ]:
df[["ricoverati_con_sintomi", "terapia_intensiva", "totale_positivi"]].describe()

## 4. Filtriamo e raggruppiamo

Possiamo tenere solo un anno e poi calcolare la media per ogni mese.

In [ ]:
dati_2021 = df[df["data"].dt.year == 2021]
dati_2021.head()

In [ ]:
media_mensile = (
    dati_2021.groupby(dati_2021["data"].dt.month)["terapia_intensiva"]
    .mean()
)
media_mensile

## 5. Disegniamo due grafici

Un grafico trasforma i numeri in una forma più facile da osservare.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(df["data"], df["terapia_intensiva"], color="tab:red")
plt.title("Terapia intensiva nel tempo")
plt.xlabel("Data")
plt.ylabel("Persone in terapia intensiva")
plt.grid()
plt.show()

In [ ]:
puliti = df[["ricoverati_con_sintomi", "terapia_intensiva"]].dropna()

plt.figure(figsize=(6, 5))
plt.scatter(puliti["ricoverati_con_sintomi"], puliti["terapia_intensiva"], alpha=0.5)
plt.title("Ricoverati e terapia intensiva")
plt.xlabel("Ricoverati con sintomi")
plt.ylabel("Terapia intensiva")
plt.grid()
plt.show()

## 6. Un primo modello di apprendimento automatico

Creiamo un modello lineare molto semplice: prova a stimare il numero in terapia intensiva usando il numero di ricoverati con sintomi. Separiamo alcuni dati per controllare il modello su esempi che non ha visto.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

dati_modello = df[["ricoverati_con_sintomi", "terapia_intensiva"]].dropna()
X = dati_modello[["ricoverati_con_sintomi"]]
y = dati_modello["terapia_intensiva"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
modello = LinearRegression()
modello.fit(X_train, y_train)
previsioni = modello.predict(X_test)
print("R² del modello:", round(r2_score(y_test, previsioni), 3))

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, previsioni, alpha=0.6)
limite = max(y_test.max(), previsioni.max())
plt.plot([0, limite], [0, limite], color="black", linestyle="--")
plt.title("Valori reali e previsioni")
plt.xlabel("Terapia intensiva reale")
plt.ylabel("Terapia intensiva prevista")
plt.grid()
plt.show()

### Leggiamo il risultato con prudenza

Il valore R² indica quanto il modello segue i dati di prova: non è una garanzia per il futuro. Una correlazione non dimostra una causa. Il modello non conosce età, cure, vaccini, varianti, ritardi di registrazione o decisioni sanitarie. Anche la qualità e il significato dei dati influenzano ogni risultato.

## Prova tu

Le celle seguenti sono esercizi: cambia una piccola parte e riesegui la cella.

In [ ]:
# >>> MODIFICA QUI: prova un anno diverso, per esempio 2022
anno_scelto = 2021
df[df["data"].dt.year == anno_scelto][["data", "totale_positivi"]].head()

In [ ]:
# >>> MODIFICA QUI: sostituisci "terapia_intensiva" con "ricoverati_con_sintomi"
colonna_da_grafico = "terapia_intensiva"
plt.figure(figsize=(10, 4))
plt.plot(df["data"], df[colonna_da_grafico])
plt.title(colonna_da_grafico)
plt.grid()
plt.show()